In [11]:
using Lux, DifferentialEquations, Optimization, OptimizationOptimisers, SciMLSensitivity
using Random, ComponentArrays, Statistics, Plots, CSV, DataFrames, Zygote


In [13]:
# 1. Load the data
file_path = "c:/Users/ADMIN/Downloads/Neural_Spiking_Dynamics/notebooks/1_data_generation/single_spike_noisy_data.csv"
HH_data = CSV.read(file_path, DataFrame)

# 2. Extract the relevant columns in order
df_ordered = HH_data[:, [:timestamp, :V, :n, :m, :h]]

# 3. Create the training arrays (t_train and data_train)
t_train = Float32.(df_ordered.timestamp)
data_train = Float32.(Matrix(df_ordered[:, [:V, :n, :m, :h]])')


4×1469 Matrix{Float32}:
 -65.0   -64.8363     -64.7979    …  -57.3823    -57.4143    -57.4497
   0.6     0.599784     0.599864       0.453623    0.453502    0.453245
   0.05    0.0502886    0.050523       0.112056    0.112553    0.112654
   0.32    0.320059     0.32014        0.395085    0.395061    0.39517

In [14]:
function hh_true_dynamics(u, p, t)
    V, n, m, h = u
    # Constants
    C_m, E_Na, E_K, E_L = 1.0f0, 50.0f0, -77.0f0, -54.4f0
    g_Na, g_K, g_L, I_ext = 120.0f0, 36.0f0, 0.3f0, 10.0f0

    # Rate functions
    α_n = 0.01f0 * (V + 55) / (1 - exp(-(V + 55) / 10))
    β_n = 0.125f0 * exp(-(V + 65) / 80)
    α_m = 0.1f0 * (V + 40) / (1 - exp(-(V + 40) / 10))
    β_m = 4.0f0 * exp(-(V + 65) / 18)
    α_h = 0.07f0 * exp(-(V + 65) / 20)
    β_h = 1.0f0 / (1 + exp(-(V + 35) / 10))

    # Currents
    I_Na = g_Na * m^3 * h * (V - E_Na)
    I_K  = g_K * n^4 * (V - E_K)
    I_L  = g_L * (V - E_L)

    dV = (I_ext - I_Na - I_K - I_L) / C_m
    dn = α_n * (1 - n) - β_n * n
    dm = α_m * (1 - m) - β_m * m
    dh = α_h * (1 - h) - β_h * h

    return [dV, dn, dm, dh]
end

hh_true_dynamics (generic function with 1 method)

In [15]:
u0 = [-65.0f0, 0.05f0, 0.6f0, 0.32f0]
tspan = (0.0f0, 30.0f0)
prob_true = ODEProblem(hh_true_dynamics, u0, tspan)

ODEProblem with uType Vector{Float32} and tType Float32. In-place: false
Non-trivial mass matrix: false
timespan: (0.0f0, 30.0f0)
u0: 4-element Vector{Float32}:
 -65.0
   0.05
   0.6
   0.32

In [16]:
# --- B. Universal Differential Equation Definition ---

# 1. The Neural Network (The "Missing Term" Estimator)
# We only need it to output a scalar (the missing current contribution to dV)
# Input: 4 states (V, n, m, h) -> Output: 1 scalar (Modification to dV)
nn_model = Lux.Chain(
    Lux.Dense(4 => 32, tanh),
    Lux.Dense(32 => 32, tanh),
    Lux.Dense(32 => 1) 
)

rng = Random.default_rng()
p_nn, st = Lux.setup(rng, nn_model)

((layer_1 = (weight = Float32[1.430542 -1.2234261 -0.31329527 -0.5285851; -1.0636693 -1.1448612 0.7424008 -0.7000711; … ; 0.22226545 0.3665172 -0.0051515894 -0.41045108; -0.20999506 -0.008929766 -1.2735714 -0.85856485], bias = Float32[-0.13025641, -0.11415768, -0.15128064, -0.29241908, 0.28074372, 0.056691945, 0.024069786, -0.33957332, -0.23938507, -0.46831155  …  0.1696577, 0.32883263, 0.18119794, 0.2950155, 0.08643162, -0.42653906, 0.1511097, 0.278377, -0.36365086, 0.11294049]), layer_2 = (weight = Float32[-0.30600992 0.00076772174 … 0.21083336 -0.060751792; 0.0501725 -0.44111088 … -0.16306508 0.36051148; … ; -0.21903697 0.38901544 … -0.36754465 0.01679948; -0.409722 -0.48108596 … -0.46207803 -0.35052153], bias = Float32[0.033687197, -0.11234678, 0.11817062, -0.1535161, -0.1621318, 0.09666321, -0.08353195, 0.059416078, -0.0874403, 0.16021974  …  0.09585509, -0.07839611, -0.11114868, -0.15976909, 0.020422382, -0.15192759, -0.12256542, 0.12980819, 0.086603895, 0.13614663]), layer_3 = (

In [17]:
# 2. The Hybrid Dynamics: f_known + NN
function ude_dynamics(u, p, t)
    V, n, m, h = u
    # Hard-coded Known Physics (Leak + K + Gating)
    C_m, E_K, E_L = 1.0f0, -77.0f0, -54.4f0
    g_K, g_L, I_ext = 36.0f0, 0.3f0, 10.0f0

    # Known Currents
    I_K = g_K * n^4 * (V - E_K)
    I_L = g_L * (V - E_L)
    
    # Gating Dynamics (Assumed Known)
    α_n = 0.01f0 * (V + 55) / (1 - exp(-(V + 55) / 10))
    β_n = 0.125f0 * exp(-(V + 65) / 80)
    α_m = 0.1f0 * (V + 40) / (1 - exp(-(V + 40) / 10))
    β_m = 4.0f0 * exp(-(V + 65) / 18)
    α_h = 0.07f0 * exp(-(V + 65) / 20)
    β_h = 1.0f0 / (1 + exp(-(V + 35) / 10))

    dn = α_n * (1 - n) - β_n * n
    dm = α_m * (1 - m) - β_m * m
    dh = α_h * (1 - h) - β_h * h

    # --- THE UDE INJECTION ---
    # The NN predicts the "Missing" term for dV
    # Ideally, NN(u) should learn: (-I_Na / C_m)
    du_missing = first(nn_model(u, p, st)[1])
    
    # dV_known only includes I_ext, I_K, and I_L
    dV_known = (I_ext - I_K - I_L) / C_m
    
    # Total dV = Known + Learned
    dV = dV_known + du_missing

    return [dV, dn, dm, dh]
end

ude_dynamics (generic function with 1 method)

In [18]:
# --- C. Training ---

p_init = ComponentArray(p_nn)
prob_ude = ODEProblem(ude_dynamics, u0, tspan, p_init)

function predict_ude(θ)
    # Solve with current NN weights
    # Use low tolerance for gradient stability
    Array(solve(prob_ude, Heun(), p=θ, saveat=t_train, 
                sensealg=InterpolatingAdjoint(autojacvec=ZygoteVJP())))
end

predict_ude (generic function with 1 method)

In [19]:
function loss_function(θ, _)
    pred = predict_ude(θ)
    # Simple MSE Loss
    loss = mean(abs2, pred .- data_train)
    return loss
end

loss_function (generic function with 1 method)

In [20]:
# Optimization
adtype = Optimization.AutoZygote()
optf = OptimizationFunction(loss_function, adtype)
optprob = OptimizationProblem(optf, p_init)

callback = function (p, l)
    println("Loss: $l")
    return false
end

#26 (generic function with 1 method)

In [ ]:
# Train
println("Training UDE...")
# 1. Fast convergence with ADAM
res1 = solve(optprob, Adam(0.05), maxiters=300, callback=callback)
# 2. Fine tuning with BFGS
optprob2 = remake(optprob, u0=res1.u)
res2 = solve(optprob2, BFGS(), maxiters=100, callback=callback)

Training UDE...
Loss: 122.295586
Loss: 112.25364
Loss: 109.43061
Loss: 108.56105
Loss: 108.24171
Loss: 108.17363
Loss: 108.2649
Loss: 108.42399
Loss: 108.57415
Loss: 108.66642
Loss: 108.6824
Loss: 108.628105
Loss: 108.52556
Loss: 108.40474
Loss: 108.29255
Loss: 108.21157
Loss: 108.17397
Loss: 108.179596
Loss: 108.21659
Loss: 108.265915
Loss: 108.30766
Loss: 108.32738
Loss: 108.32029
Loss: 108.29089
Loss: 108.25019
Loss: 108.21086
Loss: 108.18305
Loss: 108.171684
Loss: 108.17597
Loss: 108.19034
Loss: 108.207466
Loss: 108.220535
Loss: 108.22518
Loss: 108.220406
Loss: 108.20876
Loss: 108.19422
Loss: 108.18114
Loss: 108.173164
Loss: 108.17164
Loss: 108.17555
Loss: 108.182175
Loss: 108.1882
Loss: 108.191086
Loss: 108.18996
Loss: 108.185524
Loss: 108.179665
Loss: 108.17459
Loss: 108.171814
Loss: 108.17169
Loss: 108.17362
Loss: 108.17631
Loss: 108.17833
Loss: 108.17896
Loss: 108.177895
Loss: 108.17582
Loss: 108.1735
Loss: 108.17195
Loss: 108.171425
Loss: 108.17196
Loss: 108.17303
Loss: 108.17

In [ ]:
p_trained = res2.u
println("Training Complete.")
